# Fine-tune Gemma with LoRA on the loan approval instruction dataset

This Colab notebook runs the full demo from repository setup through LoRA fine-tuning, inference, and output comparison.

**What this notebook does**

- Uses the existing repository scripts (`src/train_lora.py`, `src/inference.py`, and `src/compare_outputs.py`).
- Fine-tunes a LoRA adapter on top of Gemma using the prepared loan approval JSONL files.
- Saves the adapter locally in the notebook runtime under `models/loan-approval-gemma-lora`.
- Does **not** commit or upload trained model files.

> **Before you start:** In Colab, choose **Runtime → Change runtime type → GPU**.

In [ ]:
# Replace <REPOSITORY_URL> with your GitHub repository URL before running this cell.
# Example: !git clone https://github.com/YOUR_USERNAME/testeslm.git
!git clone <REPOSITORY_URL>
%cd testeslm/slm-transactional-finetuning-demo


## 1. Check GPU availability

LoRA fine-tuning a Gemma model is intended to run on a CUDA GPU. This cell prints the available GPU and CUDA details.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")
else:
    print("No CUDA GPU detected. In Colab, enable a GPU from Runtime > Change runtime type > Hardware accelerator.")

## 2. Install dependencies

Install the Python packages used by the dataset, training, inference, and comparison scripts.

In [ ]:
!pip install -r requirements.txt

## 3. Log in to Hugging Face

Gemma models are hosted on Hugging Face. You may need to authenticate with a Hugging Face token before downloading the model.

**Gemma license note:** Some Gemma checkpoints require accepting the model terms/license on the Hugging Face model page before access is granted. If model loading fails with a gated-repository or authorization error, open the model page for `google/gemma-2-2b-it`, accept the terms, then rerun the login/model-loading cells.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 4. Confirm the prepared JSONL files exist

This demo assumes the repository already contains the prepared instruction-tuning files:

- `data/processed/train.jsonl`
- `data/processed/test.jsonl`

The next cell checks that both files are present and shows the number of rows in each file.

In [ ]:
from pathlib import Path

required_files = [
    Path("data/processed/train.jsonl"),
    Path("data/processed/test.jsonl"),
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    row_count = sum(1 for line in path.open("r", encoding="utf-8") if line.strip())
    print(f"{path}: {row_count} rows")

## 5. Run LoRA fine-tuning with demo-friendly settings

This cell calls the existing training script rather than duplicating training logic in the notebook.

The LoRA adapter will be generated locally in this Colab runtime at:

```text
models/loan-approval-gemma-lora
```

The settings below are intentionally small for a demo. Increase epochs or sequence length later if you want a longer training run.

In [ ]:
!python src/train_lora.py \
  --train-file data/processed/train.jsonl \
  --eval-file data/processed/test.jsonl \
  --output-dir models/loan-approval-gemma-lora \
  --num-train-epochs 0.25 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 4 \
  --learning-rate 2e-4 \
  --max-seq-length 512


## 6. Run inference with the base Gemma model

This provides a baseline response before attaching the fine-tuned LoRA adapter.

In [ ]:
INSTRUCTION = "Analyze the loan application based on the company's historical credit decisions."
INPUT_TEXT = "Applicant has 2 dependents, is Graduate, self-employed status is No, annual income is 9600000, requested loan amount is 29900000, loan term is 12 months, CIBIL score is 778, residential assets value is 2400000, commercial assets value is 17600000, luxury assets value is 22700000, and bank asset value is 8000000."

!python src/inference.py \
  --instruction "$INSTRUCTION" \
  --input "$INPUT_TEXT" \
  --max-new-tokens 192


## 7. Run inference with the fine-tuned LoRA adapter

This uses the same prompt, but loads the local adapter from `models/loan-approval-gemma-lora`.

In [ ]:
!python src/inference.py \
  --adapter-dir models/loan-approval-gemma-lora \
  --instruction "$INSTRUCTION" \
  --input "$INPUT_TEXT" \
  --max-new-tokens 192


## 8. Compare base and fine-tuned outputs

The comparison script runs several sample examples and writes a Markdown report to `outputs/comparison.md`.

In [ ]:
!python src/compare_outputs.py \
  --sample-file data/processed/sample.jsonl \
  --adapter-dir models/loan-approval-gemma-lora \
  --output-file outputs/comparison.md \
  --num-examples 3 \
  --max-new-tokens 192


## 9. Display the comparison report

If `outputs/comparison.md` exists, display it directly in the notebook.

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

comparison_path = Path("outputs/comparison.md")
if comparison_path.exists():
    display(Markdown(comparison_path.read_text(encoding="utf-8")))
else:
    print("outputs/comparison.md does not exist yet. Run the comparison cell above first.")

## Troubleshooting

### No GPU available

- In Colab, go to **Runtime → Change runtime type → Hardware accelerator → GPU**.
- Rerun the GPU check cell after reconnecting.
- CPU-only training will be very slow and is not recommended for this demo.

### Hugging Face authentication failure

- Create or copy a Hugging Face access token from your Hugging Face account settings.
- Rerun the `notebook_login()` cell and paste the token when prompted.
- Make sure the token has permission to read gated models.

### Gemma license not accepted

- Open the Hugging Face model page for `google/gemma-2-2b-it`.
- Accept the model terms/license while signed in to the same Hugging Face account used in Colab.
- Rerun the login and training/inference cells after access is granted.

### CUDA out-of-memory

- Restart the runtime to clear GPU memory.
- Keep `--per-device-train-batch-size 1`.
- Lower `--max-seq-length` to `384` or `256`.
- Reduce `--gradient-accumulation-steps` if memory pressure persists.
- Close other notebooks using the same GPU.

### Slow training

- Confirm that `torch.cuda.is_available()` is `True`.
- Use a smaller `--num-train-epochs` value for quick demos.
- Use a smaller `--max-seq-length` for faster iterations.
- Colab free-tier GPU type and availability can vary, so runtime can differ across sessions.